In [1]:
# Initialise Spark session.
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/21 02:47:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Run this right after creating your 'spark' session to limit log to errors.
spark.sparkContext.setLogLevel("ERROR")

In [3]:
spark.version

'4.1.1'

In [17]:
!mkdir homework/01-inputs

In [22]:
!wget -O ./homework/01-inputs/yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-21 03:20:04--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.187, 18.155.128.222, 18.155.128.46, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.187|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘./homework/01-inputs/yellow_tripdata_2025-11.parquet’

./homework/01-input 100%[===================>]  67.84M   248MB/s    in 0.3s    

2026-03-21 03:20:05 (248 MB/s) - ‘./homework/01-inputs/yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [5]:
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [23]:
!mkdir homework/02-repartition

In [25]:
df_yellow.repartition(4).write.parquet('homework/02-repartition', mode='overwrite')

In [32]:
df_yellow.createOrReplaceTempView('yellow')

In [33]:
df_q3 = spark.sql("""
select
    count(1)
from
    yellow
where date_trunc('day', tpep_pickup_datetime) = '2025-11-15';
""")

In [34]:
df_q3.show()

[Stage 19:>                                                         (0 + 4) / 4]

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [35]:
df_q4 = spark.sql("""
select
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime))/3600 as trip_length_h
from
    yellow
Order by
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime))/3600 desc
Limit 2;
""")

In [36]:
df_q4.show()

+-----------------+
|    trip_length_h|
+-----------------+
|90.64666666666666|
|76.94833333333334|
+-----------------+



In [28]:
!wget -O ./homework/01-inputs/taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-21 03:24:31--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.187, 18.155.128.222, 18.155.128.46, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.187|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘./homework/01-inputs/taxi_zone_lookup.csv’

./homework/01-input 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-21 03:24:32 (102 MB/s) - ‘./homework/01-inputs/taxi_zone_lookup.csv’ saved [12331/12331]



In [29]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('homework/01-inputs/taxi_zone_lookup.csv')

In [37]:
df_zones.createOrReplaceTempView('zones')

In [38]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [46]:
df_q6 = spark.sql("""
select
    zones.Zone,
    count(*)
from
    data
    Left Join zones
    On data.PULocationID = zones.LocationID
Where
    date_trunc('month', tpep_pickup_datetime) = '2025-11-01'
Group By
    1
Order by
    count(*)
Limit 10;
""")

In [49]:
df_q6.show(10, False)

[Stage 41:==============>                                           (1 + 3) / 4]

+---------------------------------------------+--------+
|Zone                                         |count(1)|
+---------------------------------------------+--------+
|Governor's Island/Ellis Island/Liberty Island|1       |
|Eltingville/Annadale/Prince's Bay            |1       |
|Arden Heights                                |1       |
|Port Richmond                                |3       |
|Rikers Island                                |4       |
|Rossville/Woodrow                            |4       |
|Great Kills                                  |4       |
|Green-Wood Cemetery                          |4       |
|Jamaica Bay                                  |5       |
|Westerleigh                                  |12      |
+---------------------------------------------+--------+

